# TopoMT: Comparativa de Métodos de Detección de Pockets

Este notebook muestra cómo utilizar la API unificada `get_topography` para caracterizar los bolsillos de una proteína (1TCD) utilizando diferentes algoritmos nativos integrados en **TopoMT**.

### Objetivos:
1. Cargar un sistema molecular por su PDB ID.
2. Ejecutar los métodos: `pocketeer`, `alphaspace2`, `fpocket`, `castp` y `pycasta`.
3. Comparar resultados (número de pockets, centros y volúmenes).

In [ ]:
import topomt as tmt
import molsysmt as msm
import numpy as np
import pandas as pd

# Configuramos el sistema de prueba
pdb_id = '1tcd'
methods = ['pocketeer', 'alphaspace2', 'fpocket', 'castp', 'pycasta']

## 1. Ejecución de la comparativa
Iteramos sobre cada método y almacenamos el objeto `Topography` resultante.

In [ ]:
results = []
topographies = {}

for method in methods:
    print(f"--- Ejecutando {method} ---")
    try:
        # La función get_topography se encarga de todo el cableado interno
        topo = tmt.get_topography(pdb_id, method=method)
        topographies[method] = topo
        
        pockets = topo.get_features(by='type', value='pocket')
        n_pockets = len(pockets)
        
        if n_pockets > 0:
            p0 = pockets[0]
            results.append({
                'Método': method,
                'Pockets': n_pockets,
                'Vol. Principal (nm^3)': round(p0.volume, 3),
                'Átomos': len(p0.atom_indices),
                'Centro': np.round(p0.center, 2).tolist()
            })
        else:
            results.append({
                'Método': method,
                'Pockets': 0,
                'Vol. Principal (nm^3)': 0,
                'Átomos': 0,
                'Centro': [0,0,0]
            })
            
    except Exception as e:
        print(f"Error en {method}: {e}")
        results.append({'Método': method, 'Pockets': 'FAILED'})

df = pd.DataFrame(results)
df

## 2. Visualización (Opcional)
Si tienes instalado `nglview`, puedes visualizar la topografía de cualquiera de los métodos sobre la estructura original.

In [ ]:
# Visualizar resultados de pocketeer
# topographies['pocketeer'].show()

## Notas sobre las unidades:
- Internamente, TopoMT ahora opera en **Nanómetros (nm)**.
- Los volúmenes se reportan en **nm³** (1 nm³ = 1000 Å³).
- Los centros están en coordenadas cartesianas (nm).